# Step 13 — BiLSTM representation similarity in OOF reranking

Validation only. No base-model training and no test evaluation. Use a Colab GPU and run top to bottom. Requires `train.csv`, `attention_variants_artifacts.zip`, `oof_span_reranker_artifacts.zip` and the original Step 10 Drive folder `MyDrive/CITS4012_A2_reranker_oof` including fold checkpoints, `oof_base_results/runs.csv` and the three `fold*_oof_top10.jsonl` files. A preflight check stops if these are missing; it does not retrain models.

Add six fixed cosine features from the shared BiLSTM: question vs candidate / expanded context / left context / right context, and bidirectional mean-of-best token similarities. Empty left/right context maps to zero. No externally pretrained embeddings are introduced.

Every OOF feature comes from the same fold model that generated that candidate. Candidate positions and scores are checked before extraction. Validation features come from the exact Step 10 final checkpoint. Compare unchanged 30-feature LambdaMART with 36-feature LambdaMART on identical candidates. Prior eight boundary/confidence additions are excluded to isolate this hypothesis. Representation caches are saved on Drive after each fold. Bootstrap intervals are exploratory and do not correct for earlier validation-based model selection.


In [1]:
import os,sys,subprocess,json,zipfile,shutil,hashlib,warnings,csv
from pathlib import Path
assert 'google.colab' in sys.modules
subprocess.run([sys.executable,'-m','pip','install','-q','gensim','emoji','nltk','scikit-learn','joblib','lightgbm'],check=True)
import torch,numpy as np,joblib,re
from google.colab import drive,files
from IPython.display import display,Markdown
drive.mount('/content/drive')
WORK=Path('/content/A2_representation_reranker'); WORK.mkdir(exist_ok=True); os.chdir(WORK)
sys.path.insert(0,str(WORK))
Path('src').mkdir(exist_ok=True); Path('data').mkdir(exist_ok=True)
OUT=WORK/'results'; OUT.mkdir(exist_ok=True)
BACKUP=Path('/content/drive/MyDrive/CITS4012_A2_representation_reranker'); BACKUP.mkdir(parents=True,exist_ok=True)
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
warnings.filterwarnings('ignore',message='X does not have valid feature names, but LGBMRanker was fitted with feature names',category=UserWarning)
def locate(name,folder):
    choices=[Path('/content')/name,Path('/content/drive/MyDrive')/folder/name]
    found=next((p for p in choices if p.exists()),None)
    if found is None:
        print('Upload',name); up=files.upload(); assert name in up
        found=WORK/name; found.write_bytes(up[name])
    return found

SOURCE=locate('attention_variants_artifacts.zip','CITS4012_A2_attention_variants')
OOFZIP=locate('oof_span_reranker_artifacts.zip','CITS4012_A2_reranker_oof')
FOLD_ROOT=Path('/content/drive/MyDrive/CITS4012_A2_reranker_oof')
assert (FOLD_ROOT/'oof_base_results/runs.csv').exists(), 'Restore Step 10 Drive folder: fold checkpoints, runs.csv and candidate caches are required.'


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
target=WORK/"data/train.csv"
if not target.exists():
    source=next((p for p in (Path("/content/train.csv"),Path("/content/data/train.csv")) if p.exists()),None)
    if source: shutil.copy2(source,target)
    else:
        from google.colab import files
        up=files.upload(); assert "train.csv" in up; target.write_bytes(up["train.csv"])
sha=hashlib.sha1(target.read_bytes().replace(b"\r\n",b"\n")).hexdigest()
assert sha=="8a52bb7dff56b11fdad1ab6fd3b848fbadc22849"; print("DATA READY",sha)


DATA READY 8a52bb7dff56b11fdad1ab6fd3b848fbadc22849


In [3]:
%%writefile src/__init__.py
# package marker


Writing src/__init__.py


In [4]:
%%writefile src/evaluate.py
"""Evaluation metrics for TweetQA.

Every metric is computed per example as the max over the gold answers,
and corpus scores are the mean x 100.

- BLEU-1: nltk sentence_bleu, weights (1,0,0,0), smoothing method1
- METEOR: nltk meteor_score (needs WordNet)
- ROUGE-L: our own LCS implementation, F-measure with beta = 1.2 (the
  coco-caption convention used by the official TweetQA evaluation)
- EM and token-F1: SQuAD-style, computed on normalised text
"""
import re
import string
import unicodedata
from collections import Counter

METRICS = ("bleu1", "meteor", "rougeL", "em", "f1")
_ARTICLES = re.compile(r"\b(a|an|the)\b")
_nltk_ready = False


def _ensure_nltk():
    global _nltk_ready
    if _nltk_ready:
        return
    import nltk
    for pkg, path in [("wordnet", "corpora/wordnet"), ("omw-1.4", "corpora/omw-1.4")]:
        try:
            nltk.data.find(path)
        except LookupError:
            nltk.download(pkg, quiet=True)
    _nltk_ready = True


def normalize_answer(s):
    """Lowercase, replace ASCII and Unicode punctuation with spaces,
    drop articles, collapse whitespace."""
    s = s.lower()
    s = "".join(" " if (ch in string.punctuation or unicodedata.category(ch).startswith("P")) else ch
                for ch in s)
    s = _ARTICLES.sub(" ", s)
    return " ".join(s.split())


def norm_tokens(s):
    return normalize_answer(s).split()


# ---------------------------------------------------------------- metrics

def exact_match(pred, gold):
    return float(normalize_answer(pred) == normalize_answer(gold))


def token_f1(pred, gold):
    p, g = norm_tokens(pred), norm_tokens(gold)
    if not p or not g:
        return float(p == g)
    common = Counter(p) & Counter(g)
    overlap = sum(common.values())
    if overlap == 0:
        return 0.0
    prec, rec = overlap / len(p), overlap / len(g)
    return 2 * prec * rec / (prec + rec)


def lcs_len(a, b):
    """Length of the longest common subsequence of two token lists (O(len(a)*len(b)) DP)."""
    if not a or not b:
        return 0
    prev = [0] * (len(b) + 1)
    for x in a:
        cur = [0] * (len(b) + 1)
        for j, y in enumerate(b, 1):
            cur[j] = prev[j - 1] + 1 if x == y else max(prev[j], cur[j - 1])
        prev = cur
    return prev[-1]


def rouge_l(pred, gold, beta=1.2):
    p, g = norm_tokens(pred), norm_tokens(gold)
    if not p or not g:
        return float(p == g and bool(p))
    lcs = lcs_len(p, g)
    if lcs == 0:
        return 0.0
    prec, rec = lcs / len(p), lcs / len(g)
    return (1 + beta ** 2) * prec * rec / (rec + beta ** 2 * prec)


def bleu1(pred, gold):
    from nltk.translate.bleu_score import SmoothingFunction, sentence_bleu
    p, g = norm_tokens(pred), norm_tokens(gold)
    if not p or not g:
        return 0.0
    return sentence_bleu([g], p, weights=(1, 0, 0, 0),
                         smoothing_function=SmoothingFunction().method1)


def meteor(pred, gold):
    _ensure_nltk()
    from nltk.translate.meteor_score import meteor_score
    p, g = norm_tokens(pred), norm_tokens(gold)
    if not p or not g:
        return 0.0
    return meteor_score([g], p)


_FUNCS = {"bleu1": bleu1, "meteor": meteor, "rougeL": rouge_l, "em": exact_match, "f1": token_f1}


def score_example(pred, golds, metrics=METRICS):
    """Per-example scores in [0, 1]: max over gold answers."""
    pred = pred or ""
    return {m: max(_FUNCS[m](pred, g) for g in golds) for m in metrics}


def evaluate(preds, golds_list, metrics=METRICS, return_per_example=False):
    """preds: list[str]; golds_list: list[list[str]]. Returns corpus scores (mean x 100)."""
    assert len(preds) == len(golds_list), (len(preds), len(golds_list))
    per = [score_example(p, g, metrics) for p, g in zip(preds, golds_list)]
    n = max(1, len(per))
    corpus = {m: 100.0 * sum(s[m] for s in per) / n for m in metrics}
    return (corpus, per) if return_per_example else corpus


def fmt(scores):
    return " ".join(f"{m}={scores[m]:.2f}" for m in METRICS if m in scores)


Writing src/evaluate.py


In [5]:
%%writefile src/data.py
"""TweetQA data pipeline: loading, splits, offset-preserving tokenisation,
tweet normalisation, token flags, silver-span labelling, vocabularies and
a padded DataLoader.

Design notes (see results/DATA_REPORT.md):
- test.csv is the original TweetQA dev set. It stays locked (TEST_LOCKED)
  until the final evaluation. Validation is 10% of train, grouped by tweet.
- Every token keeps its character offsets into the raw tweet, so a
  predicted span (i, j) maps back to tweet[start_i:end_j].
- The normaliser only changes the *word* used for the embedding lookup.
  The char-CNN always sees the raw (lowercased) token, so @handles stay
  distinguishable after @user -> <user>.
"""
import ast
import csv
import os
import random
import re
from collections import Counter, defaultdict
from pathlib import Path

import torch
from torch.utils.data import DataLoader, Dataset

try:
    from .evaluate import normalize_answer, rouge_l
except ImportError:  # inlined in the notebook / run as a script
    from evaluate import normalize_answer, rouge_l

TEST_LOCKED = True
DATA_DIR = Path(os.environ.get("TWEETQA_DIR", Path(__file__).resolve().parents[1] / "data"
                               if "__file__" in globals() else "data"))
SPLIT_SEED = 4012
VAL_FRAC = 0.10

# ------------------------------------------------------------------ loading

_ANSWER_ITEM = re.compile(r"'(?:[^'\\]|\\.)*'|\"(?:[^\"\\]|\\.)*\"", re.S)


def parse_answers(field):
    """The Answer column is a numpy-array repr: ['a' 'b'] (space/newline separated, mixed quotes)."""
    items = [ast.literal_eval(m) for m in _ANSWER_ITEM.findall(field)]
    if not items:
        raise ValueError(f"unparseable answer field: {field!r}")
    return [str(a) for a in items]


def _read_csv(name):
    with open(Path(DATA_DIR) / name, encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    return [dict(row_id=i, qid=r["qid"], question=r["Question"], tweet=r["Tweet"],
                 answers=parse_answers(r["Answer"]), answer_field=r["Answer"])
            for i, r in enumerate(rows)]


def _question_word(q):
    toks = tokenize(q.lower())
    w = toks[0][0] if toks else ""
    return w if w in {"what", "who", "how", "where", "why", "which", "when"} else "other"


def get_splits(seed=SPLIT_SEED, val_frac=VAL_FRAC):
    """Train/val from train.csv: exact duplicates dropped, grouped by tweet,
    stratified by the first question word of each tweet's first question."""
    rows, seen = [], set()
    for r in _read_csv("train.csv"):
        key = (r["question"], r["answer_field"], r["tweet"])
        if key not in seen:
            seen.add(key)
            rows.append(r)
    by_tweet = defaultdict(list)
    for r in rows:
        by_tweet[r["tweet"]].append(r)
    strata = defaultdict(list)
    for tweet in sorted(by_tweet):
        strata[_question_word(by_tweet[tweet][0]["question"])].append(tweet)
    rng = random.Random(seed)
    val_tweets = set()
    for s in sorted(strata):
        tweets = strata[s][:]
        rng.shuffle(tweets)
        val_tweets.update(tweets[:round(len(tweets) * val_frac)])
    train = [r for r in rows if r["tweet"] not in val_tweets]
    val = [r for r in rows if r["tweet"] in val_tweets]
    return train, val


def load_test():
    if TEST_LOCKED:
        raise RuntimeError("Test set is locked (TEST_LOCKED=True). It is used once, in Stage 8.")
    return _read_csv("test.csv")


# ------------------------------------------------------------- tokenisation

_URL = r"https?://\S+|www\.\S+"
_EMOJI_CHARS = "\U0001F000-\U0001FAFF☀-➿⬀-⯿️‍"
_TOKEN_RE = re.compile(
    rf"(?P<url>{_URL})"
    r"|(?P<mention>@\w+)"
    r"|(?P<hashtag>#\w+)"
    rf"|(?P<emoji>[{_EMOJI_CHARS}]+)"
    r"|(?P<number>\d+(?:[.,:/]\d+)*)"
    r"|(?P<word>\w+(?:['’]\w+)*)"
    r"|(?P<sym>[^\w\s])",
    re.UNICODE,
)
_ELONG = re.compile(r"(.)\1{2,}")


def tokenize(text):
    """Returns [(raw_token, start, end, kind)] with char offsets into text."""
    return [(m.group(0), m.start(), m.end(), m.lastgroup) for m in _TOKEN_RE.finditer(text)]


def normalise_tokens(raw_tokens, use_normaliser=True):
    """Map raw tokens to model tokens: dicts with word, raw, start, end, kind.

    Normaliser: @user -> <user>, url -> <url>, emoji -> <emoji>,
    #tag -> '#' + 'tag', elongation squashed to 2 repeats, lowercase.
    With use_normaliser=False the word is just the lowercased raw token.
    """
    out = []
    for raw, s, e, kind in raw_tokens:
        if not use_normaliser:
            out.append(dict(word=raw.lower(), raw=raw, start=s, end=e, kind=kind))
        elif kind == "mention":
            out.append(dict(word="<user>", raw=raw, start=s, end=e, kind=kind))
        elif kind == "url":
            out.append(dict(word="<url>", raw=raw, start=s, end=e, kind=kind))
        elif kind == "emoji":
            out.append(dict(word="<emoji>", raw=raw, start=s, end=e, kind=kind))
        elif kind == "hashtag":
            out.append(dict(word="#", raw="#", start=s, end=s + 1, kind=kind))
            tag = raw[1:]
            out.append(dict(word=_ELONG.sub(r"\1\1", tag.lower()), raw=tag, start=s + 1, end=e, kind=kind))
        else:
            out.append(dict(word=_ELONG.sub(r"\1\1", raw.lower()), raw=raw, start=s, end=e, kind=kind))
    return out


_BYLINE_RE = re.compile(r"\(@\w+\)\s+[A-Za-z]+\.?\s+\d{1,2},\s+\d{4}\s*$")
_BOUNDARY = {".", "!", "?", "—", "–", "-", "\"", "”", "|", ":"}


def byline_start(text, tokens):
    """Index of the first token of the trailing 'Name (@handle) Month D, YYYY'
    byline, or len(tokens) if there is none. The name is taken back to the
    previous sentence boundary, at most 6 tokens."""
    m = _BYLINE_RE.search(text)
    if not m:
        return len(tokens)
    k = next((i for i, t in enumerate(tokens) if t["start"] >= m.start()), len(tokens))
    j = k
    while j > 0 and k - j < 6 and tokens[j - 1]["raw"] not in _BOUNDARY:
        j -= 1
    return j


FLAG_NAMES = ("mention", "hashtag", "url", "emoji", "number", "capitalised", "in_byline", "in_other_seq")


def token_flags(tokens, other_words, byline_from=None):
    by = len(tokens) if byline_from is None else byline_from
    flags = []
    for i, t in enumerate(tokens):
        flags.append([
            float(t["kind"] == "mention"), float(t["kind"] == "hashtag"), float(t["kind"] == "url"),
            float(t["kind"] == "emoji"), float(t["kind"] == "number"),
            float(t["raw"][:1].isupper()), float(i >= by),
            float(t["word"] in other_words and t["word"] not in _STOP),
        ])
    return flags


_STOP = set("""a an the of to in on at for by with from and or but is are was were be been being am
do does did has have had i you he she it we they me him her us them my your his its our their this that
these those what who whom whose which when where why how not no yes so if than then there here as just
about into out up down over can will would should could may might must 's ? . , ! ' " - — : ; ( )""".split())


# ------------------------------------------------------------- silver spans

def _norm_toks(raw):
    return normalize_answer(raw).split()


def silver_spans(tokens, tweet, answers, max_len=15, tau=0.5):
    """Score every span (i, j), j - i < max_len, that starts and ends on a
    content token, by token-F1 against each gold answer (max over golds).

    Returns (best, best_f1, soft):
      best  : (i, j) with the highest F1, ties broken by ROUGE-L then length
      soft  : [(i, j, w)] for all spans with F1 >= tau, weights proportional to F1
    """
    ntok = [_norm_toks(t["raw"]) for t in tokens]
    content = [bool(x) for x in ntok]
    golds = [Counter(_norm_toks(a)) for a in answers]
    glens = [sum(g.values()) for g in golds]
    scored = []
    for i in range(len(tokens)):
        if not content[i]:
            continue
        bag, n = Counter(), 0
        for j in range(i, min(len(tokens), i + max_len)):
            bag.update(ntok[j])
            n += len(ntok[j])
            if not content[j]:
                continue
            f = 0.0
            for g, gl in zip(golds, glens):
                if gl == 0:
                    continue
                ov = sum((bag & g).values())
                if ov:
                    p, r = ov / n, ov / gl
                    f = max(f, 2 * p * r / (p + r))
            if f > 0:
                scored.append((f, i, j))
    if not scored:
        return None, 0.0, []
    top = max(f for f, _, _ in scored)
    ties = [(i, j) for f, i, j in scored if f == top]
    if len(ties) > 1:
        def key(ij):
            txt = tweet[tokens[ij[0]]["start"]:tokens[ij[1]]["end"]]
            return (max(rouge_l(txt, a) for a in answers), -(ij[1] - ij[0]), -ij[0])
        ties.sort(key=key, reverse=True)
    soft = [(i, j, f) for f, i, j in scored if f >= tau]
    z = sum(w for _, _, w in soft)
    soft = [(i, j, w / z) for i, j, w in soft]
    return ties[0], top, soft


def span_text(ex, i, j):
    return ex["tweet"][ex["t_tokens"][i]["start"]:ex["t_tokens"][j]["end"]]


# ------------------------------------------------------------------- vocab

class Vocab:
    PAD, UNK = "<pad>", "<unk>"

    def __init__(self, counter, min_freq=2, specials=()):
        self.itos = [self.PAD, self.UNK] + [s for s in specials if s not in (self.PAD, self.UNK)]
        self.itos += sorted([w for w, c in counter.items() if c >= min_freq and w not in self.itos],
                            key=lambda w: (-counter[w], w))
        self.stoi = {w: i for i, w in enumerate(self.itos)}

    def __len__(self):
        return len(self.itos)

    def __call__(self, w):
        return self.stoi.get(w, 1)


SPECIALS = ("<user>", "<url>", "<emoji>")
MAX_WORD_CHARS = 16


def build_vocabs(examples, min_freq=2, char_min_freq=5):
    """Vocabularies come from the training split only."""
    wc, cc = Counter(), Counter()
    for ex in examples:
        for t in ex["t_tokens"] + ex["q_tokens"]:
            wc[t["word"]] += 1
            cc.update(t["raw"].lower()[:MAX_WORD_CHARS])
    return Vocab(wc, min_freq, SPECIALS), Vocab(cc, char_min_freq)


# ----------------------------------------------------------------- encoding

# tau: an example is trainable if its best silver span has F1 >= tau.
# soft_tau: minimum F1 for a span to join the soft target set (None = same as tau). Kept separate so the
# soft-set width can be varied without changing which examples are trained on.
DEFAULT_PREP = dict(use_normaliser=True, max_span_len=15, tau=0.5, soft_tau=None)


def preprocess(rows, prep=None):
    """Tokenise, normalise, flag and silver-label raw rows."""
    prep = {**DEFAULT_PREP, **(prep or {})}
    out = []
    for r in rows:
        t_tokens = normalise_tokens(tokenize(r["tweet"]), prep["use_normaliser"])
        q_tokens = normalise_tokens(tokenize(r["question"]), prep["use_normaliser"])
        by = byline_start(r["tweet"], t_tokens)
        t_words = {t["word"] for t in t_tokens}
        q_words = {t["word"] for t in q_tokens}
        soft_tau = prep["tau"] if prep.get("soft_tau") is None else prep["soft_tau"]
        best, best_f1, soft = silver_spans(t_tokens, r["tweet"], r["answers"], prep["max_span_len"], soft_tau)
        out.append(dict(
            r, t_tokens=t_tokens, q_tokens=q_tokens, byline_from=by,
            t_flags=token_flags(t_tokens, q_words, by), q_flags=token_flags(q_tokens, t_words),
            best=best, best_f1=best_f1, soft=soft,
            trainable=best is not None and best_f1 >= prep["tau"],
        ))
    return out


def encode(examples, wvocab, cvocab):
    for ex in examples:
        for side in ("t", "q"):
            toks = ex[f"{side}_tokens"]
            ex[f"{side}_ids"] = [wvocab(t["word"]) for t in toks]
            ex[f"{side}_chars"] = [[cvocab(c) for c in t["raw"].lower()[:MAX_WORD_CHARS]] or [1] for t in toks]
    return examples


class QADataset(Dataset):
    def __init__(self, examples, train=False):
        # during training, only examples with a usable silver span contribute to the loss
        self.examples = [e for e in examples if e["trainable"]] if train else list(examples)

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, i):
        return self.examples[i]


def _pad_side(batch, side, n_flags):
    B = len(batch)
    T = max(1, max(len(ex[f"{side}_ids"]) for ex in batch))
    W = max(1, max((len(c) for ex in batch for c in ex[f"{side}_chars"]), default=1))
    words = torch.zeros(B, T, dtype=torch.long)
    chars = torch.zeros(B, T, W, dtype=torch.long)
    flags = torch.zeros(B, T, n_flags)
    mask = torch.zeros(B, T, dtype=torch.bool)
    for b, ex in enumerate(batch):
        n = len(ex[f"{side}_ids"])
        words[b, :n] = torch.tensor(ex[f"{side}_ids"], dtype=torch.long)
        for k, cs in enumerate(ex[f"{side}_chars"]):
            chars[b, k, :len(cs)] = torch.tensor(cs, dtype=torch.long)
        if n:
            flags[b, :n] = torch.tensor(ex[f"{side}_flags"])
        mask[b, :n] = True
    return words, chars, flags, mask


def collate(batch):
    tw, tc, tf, tm = _pad_side(batch, "t", len(FLAG_NAMES))
    qw, qc, qf, qm = _pad_side(batch, "q", len(FLAG_NAMES))
    B, T = tw.shape
    y_start = torch.full((B,), -1, dtype=torch.long)
    y_end = torch.full((B,), -1, dtype=torch.long)
    soft_start = torch.zeros(B, T)
    soft_end = torch.zeros(B, T)
    for b, ex in enumerate(batch):
        if ex["best"] is not None:
            y_start[b], y_end[b] = ex["best"]
        for i, j, w in ex["soft"]:
            soft_start[b, i] += w
            soft_end[b, j] += w
    return dict(t_words=tw, t_chars=tc, t_flags=tf, t_mask=tm,
                q_words=qw, q_chars=qc, q_flags=qf, q_mask=qm,
                y_start=y_start, y_end=y_end, soft_start=soft_start, soft_end=soft_end,
                best_f1=torch.tensor([ex["best_f1"] for ex in batch], dtype=torch.float),
                meta=batch)


def make_loader(examples, batch_size=32, train=False, seed=0):
    ds = QADataset(examples, train=train)
    g = torch.Generator()
    g.manual_seed(seed)
    return DataLoader(ds, batch_size=batch_size, shuffle=train, collate_fn=collate, generator=g)


# ------------------------------------------------------------------ oracle

def oracle_predictions(examples):
    """Best silver span text for every example ('' if there is no overlapping span)."""
    return [span_text(ex, *ex["best"]) if ex["best"] is not None else "" for ex in examples]


Writing src/data.py


In [6]:
%%writefile src/runlog.py
"""Append-only experiment log: every reported number must come from a row in results/runs.csv."""
import csv
import json
import os
import time
from pathlib import Path

RESULTS_DIR = Path(os.environ.get("A2_RESULTS", Path(__file__).resolve().parents[1] / "results"
                                  if "__file__" in globals() else "results"))
RUNS_CSV = RESULTS_DIR / "runs.csv"
FIELDS = ["run_id", "timestamp", "model", "config", "seed", "split",
          "bleu1", "meteor", "rougeL", "em", "f1", "train_time_s", "notes"]


def log_run(model, config, seed, split, scores, train_time_s=0.0, run_id=None, notes=""):
    RUNS_CSV.parent.mkdir(parents=True, exist_ok=True)
    new = not RUNS_CSV.exists()
    run_id = run_id or f"{model}-{split}-s{seed}-{time.strftime('%Y%m%d-%H%M%S')}"
    row = dict(run_id=run_id, timestamp=time.strftime("%Y-%m-%d %H:%M:%S"), model=model,
               config=json.dumps(config, sort_keys=True, default=str), seed=seed, split=split,
               train_time_s=round(train_time_s, 1), notes=notes,
               **{m: round(scores[m], 4) for m in ("bleu1", "meteor", "rougeL", "em", "f1")})
    with open(RUNS_CSV, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=FIELDS)
        if new:
            w.writeheader()
        w.writerow(row)
    return run_id


def read_runs():
    if not RUNS_CSV.exists():
        return []
    with open(RUNS_CSV, encoding="utf-8") as f:
        return list(csv.DictReader(f))


Writing src/runlog.py


In [7]:
%%writefile src/model.py
"""TweetBiDAF: an extractive QA model for tweets, trained from scratch.

  1. Embedding: word embedding ⊕ char-CNN over the *raw* token ⊕ token flags
     -> 2-layer highway                                    (A1: use_char_cnn, A2: use_token_flags)
  2. Shared BiLSTM contextual encoder for tweet (H) and question (U)
  3. Trilinear similarity S_ij = w^T [h_i; u_j; h_i ∘ u_j]
     C2Q = softmax_j(S) U,  Q2C = softmax_i(max_j S) H
     G = [h; c2q; h ∘ c2q; h ∘ q2c]                        (A4: biattn_mode "c2q" drops Q2C)
  4. Configurable multi-head self-attention over the projected G,
     residual + LayerNorm                                   (A3: use_self_attn)
  5. 2-layer modelling BiLSTM, start head, 1 more BiLSTM layer, end head,
     padding masks, best (i, j) with i <= j < i + L

The design follows BiDAF (Seo et al., 2017) with gated self-attention in the
style of R-Net (Wang et al., 2017). The tweet-specific parts are written for
this project: a char-CNN on raw tokens with a normalised word vocabulary,
token flags (including the byline flag), soft silver-span training, and
self-attention for long-range links between the tweet body and its byline.
"""
import torch
import torch.nn as nn
import torch.nn.functional as F

NEG = -1e30

DEFAULT_MODEL = dict(
    word_dim=100, char_dim=32, char_filters=50, char_widths=(3, 4, 5), n_flags=8,
    hidden=128, dropout=0.2, attn_heads=4, max_span_len=15,
    use_char_cnn=True, use_token_flags=True, use_self_attn=True,
    self_attn_mode="gated", biattn_mode="both",
    condition_end_on_start=False,
    use_joint_span_scorer=False, span_scorer_dim=64,
)


class CharCNN(nn.Module):
    def __init__(self, n_chars, char_dim, filters, widths):
        super().__init__()
        self.emb = nn.Embedding(n_chars, char_dim, padding_idx=0)
        self.convs = nn.ModuleList(nn.Conv1d(char_dim, filters, w, padding=w // 2) for w in widths)
        self.out_dim = filters * len(widths)

    def forward(self, chars):                      # (B, T, W)
        B, T, W = chars.shape
        x = self.emb(chars.view(B * T, W)).transpose(1, 2)            # (BT, c, W)
        cmask = (chars.view(B * T, 1, W) != 0)
        outs = []
        for conv in self.convs:
            y = F.relu(conv(x))[:, :, :W]
            y = y.masked_fill(~cmask, 0.0)          # ReLU output >= 0, so masked max-pool is safe
            outs.append(y.max(dim=2).values)
        return torch.cat(outs, -1).view(B, T, -1)


class Highway(nn.Module):
    def __init__(self, dim, n_layers=2):
        super().__init__()
        self.lin = nn.ModuleList(nn.Linear(dim, dim) for _ in range(n_layers))
        self.gate = nn.ModuleList(nn.Linear(dim, dim) for _ in range(n_layers))

    def forward(self, x):
        for lin, gate in zip(self.lin, self.gate):
            g = torch.sigmoid(gate(x))
            x = g * F.relu(lin(x)) + (1 - g) * x
        return x


def reverse_padded(x, mask):
    """Reverse each sequence within its own length; padding stays at the end."""
    T = x.size(1)
    lengths = mask.sum(1, keepdim=True)                                  # (B, 1)
    t = torch.arange(T, device=x.device).unsqueeze(0)                    # (1, T)
    idx = torch.where(t < lengths, lengths - 1 - t, t)                   # (B, T)
    return x.gather(1, idx.unsqueeze(-1).expand_as(x))


class BiLSTM(nn.Module):
    """BiLSTM over right-padded batches, padding-exact without packing.

    The forward LSTM reads valid tokens before any padding, so its outputs at
    valid positions are exact. The backward direction runs a forward LSTM over
    each sequence reversed *within its length*, then un-reverses the output.
    The result equals a packed BiLSTM, but runs ~4x faster on MPS.
    """

    def __init__(self, in_dim, hidden, layers=1, dropout=0.0):
        super().__init__()
        self.fw = nn.ModuleList(nn.LSTM(in_dim if l == 0 else 2 * hidden, hidden, batch_first=True)
                                for l in range(layers))
        self.bw = nn.ModuleList(nn.LSTM(in_dim if l == 0 else 2 * hidden, hidden, batch_first=True)
                                for l in range(layers))
        self.drop = nn.Dropout(dropout)

    def forward(self, x, mask):
        for l, (fw, bw) in enumerate(zip(self.fw, self.bw)):
            if l > 0:
                x = self.drop(x)
            f = fw(x)[0]
            b = reverse_padded(bw(reverse_padded(x, mask))[0], mask)
            x = torch.cat([f, b], -1) * mask.unsqueeze(-1)
        return x


class ResidualSelfAttention(nn.Module):
    """Standard multi-head self-attention with dropout, residual, and LayerNorm."""

    def __init__(self, dim, heads, dropout):
        super().__init__()
        self.mha = nn.MultiheadAttention(dim, heads, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, mask, question=None, question_mask=None):
        attended, w = self.mha(x, x, x, key_padding_mask=~mask,
                               need_weights=True, average_attn_weights=True)
        return self.norm(x + self.drop(attended)), w


class GatedSelfAttention(nn.Module):
    """Multi-head self-attention, then a sigmoid gate on [x; attended], then residual + LayerNorm."""

    def __init__(self, dim, heads, dropout):
        super().__init__()
        self.mha = nn.MultiheadAttention(dim, heads, dropout=dropout, batch_first=True)
        self.gate = nn.Linear(2 * dim, dim)
        self.norm = nn.LayerNorm(dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, mask, question=None, question_mask=None):
        attended, w = self.mha(x, x, x, key_padding_mask=~mask,
                               need_weights=True, average_attn_weights=True)
        g = torch.sigmoid(self.gate(torch.cat([x, attended], -1)))
        return self.norm(x + self.drop(g * attended)), w


class QuestionConditionedSelfAttention(nn.Module):
    """Gated self-attention whose queries include a masked question summary.

    The tweet remains the key/value sequence.  Adding a learned projection of
    the question to every tweet query biases attention toward tweet tokens that
    are relevant to that question, while retaining the current gated residual.
    """

    def __init__(self, dim, heads, dropout):
        super().__init__()
        self.question_proj = nn.Linear(dim, dim)
        self.mha = nn.MultiheadAttention(dim, heads, dropout=dropout, batch_first=True)
        self.gate = nn.Linear(2 * dim, dim)
        self.norm = nn.LayerNorm(dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, mask, question=None, question_mask=None):
        if question is None or question_mask is None:
            raise ValueError("question-conditioned attention requires question states and mask")
        weights = question_mask.unsqueeze(-1).to(question.dtype)
        summary = (question * weights).sum(1) / weights.sum(1).clamp_min(1.0)
        query = x + self.question_proj(summary).unsqueeze(1)
        attended, w = self.mha(query, x, x, key_padding_mask=~mask,
                               need_weights=True, average_attn_weights=True)
        g = torch.sigmoid(self.gate(torch.cat([x, attended], -1)))
        return self.norm(x + self.drop(g * attended)), w


class TweetBiDAF(nn.Module):
    def __init__(self, n_words, n_chars, cfg=None, word_vectors=None):
        super().__init__()
        self.cfg = cfg = {**DEFAULT_MODEL, **(cfg or {})}
        H, p = cfg["hidden"], cfg["dropout"]
        self.word_emb = nn.Embedding(n_words, cfg["word_dim"], padding_idx=0)
        if word_vectors is not None:
            self.word_emb.weight.data.copy_(word_vectors)
        emb_dim = cfg["word_dim"]
        if cfg["use_char_cnn"]:
            self.char_cnn = CharCNN(n_chars, cfg["char_dim"], cfg["char_filters"], cfg["char_widths"])
            emb_dim += self.char_cnn.out_dim
        if cfg["use_token_flags"]:
            emb_dim += cfg["n_flags"]
        self.highway = Highway(emb_dim, 2)
        self.drop = nn.Dropout(p)
        self.encoder = BiLSTM(emb_dim, H)
        self.w_sim = nn.Linear(6 * H, 1)           # trilinear: w^T [h; u; h∘u]
        g_dim = 8 * H if cfg["biattn_mode"] == "both" else 6 * H
        self.g_proj = nn.Linear(g_dim, 2 * H)
        if cfg["use_self_attn"]:
            modes = {
                "residual": ResidualSelfAttention,
                "gated": GatedSelfAttention,
                "question_gated": QuestionConditionedSelfAttention,
            }
            if cfg["self_attn_mode"] not in modes:
                raise ValueError(f"unknown self_attn_mode: {cfg['self_attn_mode']!r}")
            self.self_attn = modes[cfg["self_attn_mode"]](2 * H, cfg["attn_heads"], p)
        self.model1 = BiLSTM(2 * H, H, layers=2, dropout=p)
        self.model2 = BiLSTM(2 * H, H)
        self.start_head = nn.Linear(4 * H, 1)
        # Optional boundary-aware head: the predicted start distribution forms a
        # differentiable summary of the likely answer start.  Each end candidate
        # is scored jointly with that summary and their elementwise interaction.
        end_dim = 8 * H if cfg["condition_end_on_start"] else 4 * H
        self.end_head = nn.Linear(end_dim, 1)
        if cfg["use_joint_span_scorer"]:
            self.span_start = nn.Linear(2 * H, cfg["span_scorer_dim"], bias=False)
            self.span_end = nn.Linear(2 * H, cfg["span_scorer_dim"], bias=False)
            self.span_length_bias = nn.Embedding(cfg["max_span_len"], 1)

    # ---------------------------------------------------------------- layers
    def embed(self, words, chars, flags):
        parts = [self.word_emb(words)]
        if self.cfg["use_char_cnn"]:
            parts.append(self.char_cnn(chars))
        if self.cfg["use_token_flags"]:
            parts.append(flags)
        return self.highway(self.drop(torch.cat(parts, -1)))

    def similarity(self, h, u):
        """S_ij = w_h·h_i + w_u·u_j + w_hu·(h_i∘u_j), computed without materialising [h; u; h∘u]."""
        d = h.size(-1)
        w = self.w_sim.weight.squeeze(0)
        wh, wu, whu = w[:d], w[d:2 * d], w[2 * d:]
        return (h @ wh).unsqueeze(2) + (u @ wu).unsqueeze(1) + (h * whu) @ u.transpose(1, 2) + self.w_sim.bias

    def forward(self, batch, return_attn=False):
        tm, qm = batch["t_mask"], batch["q_mask"]
        h = self.drop(self.encoder(self.embed(batch["t_words"], batch["t_chars"], batch["t_flags"]), tm))
        u = self.drop(self.encoder(self.embed(batch["q_words"], batch["q_chars"], batch["q_flags"]), qm))

        S = self.similarity(h, u)                                        # (B, T, J)
        a = F.softmax(S.masked_fill(~qm.unsqueeze(1), NEG), dim=-1)      # C2Q weights
        c2q = a @ u
        parts = [h, c2q, h * c2q]
        b = None
        if self.cfg["biattn_mode"] == "both":
            smax = S.masked_fill(~qm.unsqueeze(1), NEG).max(-1).values    # (B, T)
            b = F.softmax(smax.masked_fill(~tm, NEG), dim=-1)            # Q2C weights
            q2c = (b.unsqueeze(1) @ h).expand_as(h)
            parts.append(h * q2c)
        x = self.drop(F.relu(self.g_proj(torch.cat(parts, -1))))

        sa = None
        if self.cfg["use_self_attn"]:
            x, sa = self.self_attn(x, tm, u, qm)

        m1 = self.drop(self.model1(x, tm))
        m2 = self.drop(self.model2(m1, tm))
        start = self.start_head(torch.cat([x, m1], -1)).squeeze(-1).masked_fill(~tm, NEG)
        if self.cfg["condition_end_on_start"]:
            start_prob = F.softmax(start, dim=-1)
            start_summary = torch.bmm(start_prob.unsqueeze(1), m1).expand_as(m2)
            end_features = torch.cat([x, m2, start_summary, m2 * start_summary], -1)
        else:
            start_prob = None
            end_features = torch.cat([x, m2], -1)
        end = self.end_head(end_features).squeeze(-1).masked_fill(~tm, NEG)
        span_logits = None
        if self.cfg["use_joint_span_scorer"]:
            ps = self.span_start(m1)
            pe = self.span_end(m2)
            pair = torch.einsum("bih,bjh->bij", ps, pe) / (ps.size(-1) ** 0.5)
            T = start.size(1)
            idx = torch.arange(T, device=start.device)
            length = idx.unsqueeze(0) - idx.unsqueeze(1)
            valid = ((length >= 0) & (length < self.cfg["max_span_len"]))
            valid = valid.unsqueeze(0) & tm.unsqueeze(2) & tm.unsqueeze(1)
            length_bias = self.span_length_bias(length.clamp(0, self.cfg["max_span_len"] - 1)).squeeze(-1)
            span_logits = (start.unsqueeze(2) + end.unsqueeze(1) + pair + length_bias).masked_fill(~valid, NEG)
        out = dict(start_logits=start, end_logits=end, span_logits=span_logits)
        if return_attn:
            out.update(S=S, c2q=a, q2c=b, self_attn=sa,
                       start_prob_for_end=start_prob)
        return out


# ------------------------------------------------------------------ loss / decode

def span_loss(out, batch, soft=True, cfg=None):
    cfg = cfg or {}
    ls = F.log_softmax(out["start_logits"], -1)
    le = F.log_softmax(out["end_logits"], -1)
    if soft:
        per_example = -(batch["soft_start"] * ls).sum(-1) - (batch["soft_end"] * le).sum(-1)
    else:
        per_example = F.nll_loss(ls, batch["y_start"], reduction="none") + F.nll_loss(le, batch["y_end"], reduction="none")
    weights = batch["best_f1"] if cfg.get("use_quality_weight", False) else torch.ones_like(per_example)
    loss = (per_example * weights).sum() / weights.sum().clamp_min(1e-8)

    if out.get("span_logits") is not None and cfg.get("use_joint_span_loss", False):
        B, T, _ = out["span_logits"].shape
        flat = out["span_logits"].reshape(B, -1)
        gold_idx = batch["y_start"] * T + batch["y_end"]
        joint = F.cross_entropy(flat, gold_idx, reduction="none")
        loss = loss + cfg.get("joint_span_weight", 0.5) * (joint * weights).sum() / weights.sum().clamp_min(1e-8)
        if cfg.get("use_hard_negative", False):
            gold_score = flat.gather(1, gold_idx.unsqueeze(1)).squeeze(1)
            negatives = flat.clone()
            negatives.scatter_(1, gold_idx.unsqueeze(1), NEG)
            hardest = negatives.max(1).values
            margin = F.relu(cfg.get("hard_negative_margin", 1.0) - gold_score + hardest)
            loss = loss + cfg.get("hard_negative_weight", 0.2) * (margin * weights).sum() / weights.sum().clamp_min(1e-8)
    return loss


@torch.no_grad()
def decode(start_logits, end_logits, max_len=15):
    """argmax over i <= j < i + max_len of log p_s(i) + log p_e(j). Returns (B, 2) indices and scores."""
    ls = F.log_softmax(start_logits, -1)
    le = F.log_softmax(end_logits, -1)
    T = ls.size(1)
    scores = ls.unsqueeze(2) + le.unsqueeze(1)                           # (B, T, T)
    idx = torch.arange(T, device=ls.device)
    band = (idx.unsqueeze(0) >= idx.unsqueeze(1)) & (idx.unsqueeze(0) - idx.unsqueeze(1) < max_len)
    scores = scores.masked_fill(~band, NEG)
    flat = scores.view(scores.size(0), -1)
    best = flat.argmax(-1)
    return torch.stack([best // T, best % T], -1), flat.max(-1).values


@torch.no_grad()
def decode_joint(span_logits):
    """Decode a globally scored span matrix whose invalid entries are already masked."""
    B, T, _ = span_logits.shape
    flat = span_logits.reshape(B, -1)
    best = flat.argmax(-1)
    return torch.stack([best // T, best % T], -1), flat.max(-1).values


Writing src/model.py


In [8]:
%%writefile src/fasttext_init.py
"""Optional in-domain word vectors: FastText (gensim) trained from scratch on the
*training split's* tweets and questions only. No external text and no
pretrained weights, so the main model is still trained from scratch on
TweetQA alone."""
import torch


def fasttext_vectors(wv, cfg, train_examples, epochs=20, seed=1):
    from gensim.models import FastText
    sents, seen = [], set()
    for ex in train_examples:
        if ex["tweet"] not in seen:                 # each tweet once
            seen.add(ex["tweet"])
            sents.append([t["word"] for t in ex["t_tokens"]])
        sents.append([t["word"] for t in ex["q_tokens"]])
    ft = FastText(vector_size=cfg["word_dim"], window=5, min_count=1, sg=1, epochs=epochs,
                  seed=seed, workers=1, min_n=3, max_n=5)
    ft.build_vocab(sents)
    ft.train(sents, total_examples=len(sents), epochs=epochs)
    W = torch.randn(len(wv), cfg["word_dim"]) * 0.1
    W[0] = 0.0
    for i, w in enumerate(wv.itos[2:], start=2):
        W[i] = torch.tensor(ft.wv[w])               # subword n-grams also give OOV-safe vectors
    return W


Writing src/fasttext_init.py


In [9]:
%%writefile src/train.py
"""Training loop for TweetBiDAF: config dict -> train / early stop / checkpoint / runs.csv.

Usage (from A2/):
  python -m src.train --name main --seed 1                        # full run
  python -m src.train --name smoke --smoke                        # 200 examples, 2 epochs
  python -m src.train --name a3 --set use_self_attn=false --seed 2
"""
import argparse
import json
import os
import random
import sys
import time
from pathlib import Path

import numpy as np
import torch

try:
    from . import data as D
    from .evaluate import evaluate, fmt
    from .model import TweetBiDAF, decode, decode_joint, span_loss
    from .runlog import RESULTS_DIR, log_run
except ImportError:
    import data as D
    from evaluate import evaluate, fmt
    from model import TweetBiDAF, decode, decode_joint, span_loss
    from runlog import RESULTS_DIR, log_run

DEFAULT_TRAIN = dict(
    lr=1e-3, batch_size=32, epochs=30, patience=4, grad_clip=5.0, weight_decay=0.0,
    use_normaliser=True, tau=0.5, soft_tau=None, soft_spans=True, min_freq=2, word_init="random",
    max_span_len=15,
    use_quality_weight=False, use_joint_span_loss=False, joint_span_weight=0.5,
    use_hard_negative=False, hard_negative_weight=0.2, hard_negative_margin=1.0,
)
CKPT_DIR = Path(os.environ.get("A2_CKPT", RESULTS_DIR.parent / "checkpoints"))


def get_device():
    return "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    if hasattr(torch, "mps") and torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)


def to_device(batch, device):
    return {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in batch.items()}


def prepare_data(cfg, train_rows=None, val_rows=None, limit=None):
    """Preprocess and encode the train/val splits. Vocab comes from the training split only."""
    if train_rows is None:
        train_rows, val_rows = D.get_splits()
    if limit:
        train_rows, val_rows = train_rows[:limit], val_rows[:limit]
    prep = dict(use_normaliser=cfg["use_normaliser"], max_span_len=cfg["max_span_len"], tau=cfg["tau"],
                soft_tau=cfg.get("soft_tau"))
    tr, va = D.preprocess(train_rows, prep), D.preprocess(val_rows, prep)
    wv, cv = D.build_vocabs(tr, min_freq=cfg["min_freq"])
    D.encode(tr, wv, cv)
    D.encode(va, wv, cv)
    return tr, va, wv, cv


@torch.no_grad()
def predict(model, examples, device, batch_size=64, max_span_len=15):
    model.eval()
    preds, spans = [], []
    for batch in D.make_loader(examples, batch_size=batch_size):
        meta = batch["meta"]
        out = model(to_device(batch, device))
        idx, _ = (decode_joint(out["span_logits"]) if out.get("span_logits") is not None
                  else decode(out["start_logits"], out["end_logits"], max_span_len))
        for ex, (i, j) in zip(meta, idx.tolist()):
            spans.append((i, j))
            preds.append(D.span_text(ex, i, j))
    return preds, spans


def build_model(cfg, wv, cv, word_vectors=None):
    mcfg = {k: cfg[k] for k in _default_model() if k in cfg}
    return TweetBiDAF(len(wv), len(cv), mcfg, word_vectors)


def train(cfg=None, seed=1, name="main", limit=None, data=None, log=print, save=True):
    """Train one model. Returns dict(best scores, history, checkpoint path, run_id)."""
    cfg = {**DEFAULT_TRAIN, **_default_model(), **(cfg or {})}
    seed_everything(seed)
    device = get_device()
    tr, va, wv, cv = data if data is not None else prepare_data(cfg, limit=limit)
    wvecs = None
    if cfg["word_init"] == "fasttext":
        try:
            from .fasttext_init import fasttext_vectors
        except ImportError:
            from fasttext_init import fasttext_vectors
        wvecs = fasttext_vectors(wv, cfg, tr, seed=seed)
    model = build_model(cfg, wv, cv, wvecs).to(device)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    opt = torch.optim.Adam(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    loader = D.make_loader(tr, batch_size=cfg["batch_size"], train=True, seed=seed)
    golds = [e["answers"] for e in va]
    run_id = f"{name}-s{seed}-{time.strftime('%Y%m%d-%H%M%S')}"
    log(f"[{run_id}] device={device} params={n_params:,} train_used={len(loader.dataset)}/{len(tr)} "
        f"val={len(va)} vocab={len(wv)} chars={len(cv)}")
    log(f"[{run_id}] config={json.dumps(cfg, sort_keys=True)}")

    best, best_epoch, bad, history, best_preds = None, -1, 0, [], []
    ckpt = CKPT_DIR / f"{run_id}.pt"
    t_start = time.time()
    for epoch in range(1, cfg["epochs"] + 1):
        model.train()
        t0, tot, n = time.time(), 0.0, 0
        for batch in loader:
            batch = to_device(batch, device)
            loss = span_loss(model(batch), batch, soft=cfg["soft_spans"], cfg=cfg)
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"])
            opt.step()
            tot += loss.item() * batch["t_words"].size(0)
            n += batch["t_words"].size(0)
        train_time = time.time() - t0
        preds, spans = predict(model, va, device, max_span_len=cfg["max_span_len"])
        scores = evaluate(preds, golds)
        history.append(dict(epoch=epoch, train_loss=tot / max(1, n), epoch_time=train_time, **scores))
        improved = best is None or scores["rougeL"] > best["rougeL"]
        log(f"[{run_id}] epoch {epoch:2d} loss={tot / max(1, n):.4f} time={train_time:.1f}s val {fmt(scores)}"
            + (" *" if improved else ""))
        if improved:
            best, best_epoch, bad = scores, epoch, 0
            best_preds = [dict(row_id=e["row_id"], qid=e["qid"], pred=p, span=sp)
                          for e, p, sp in zip(va, preds, spans)]
            if save:
                CKPT_DIR.mkdir(parents=True, exist_ok=True)
                torch.save(dict(state_dict=model.state_dict(), cfg=cfg, seed=seed, epoch=epoch,
                                vocab=wv.itos, chars=cv.itos), ckpt)
        else:
            bad += 1
            if bad >= cfg["patience"]:
                log(f"[{run_id}] early stop at epoch {epoch} (best epoch {best_epoch})")
                break
    total = time.time() - t_start
    log(f"[{run_id}] best epoch {best_epoch}: val {fmt(best)} | total {total/60:.1f} min")
    if save:   # smoke tests (save=False) leave no files behind, so results/ only holds logged runs
        hist_dir = RESULTS_DIR / "history"
        hist_dir.mkdir(parents=True, exist_ok=True)
        (hist_dir / f"{run_id}.json").write_text(json.dumps(dict(cfg=cfg, seed=seed, history=history,
                                                                 best_epoch=best_epoch), indent=1))
        pred_dir = RESULTS_DIR / "preds"
        pred_dir.mkdir(parents=True, exist_ok=True)
        (pred_dir / f"{run_id}.json").write_text(json.dumps(best_preds))
        log_run(name, cfg, seed, "val", best, total, run_id=run_id,
                notes=f"best_epoch={best_epoch} params={n_params} limit={limit} device={device}"
                + (f":{torch.cuda.get_device_name(0)}" if device == "cuda" else ""))
    return dict(best=best, best_epoch=best_epoch, history=history, ckpt=str(ckpt), run_id=run_id,
                model=model, data=(tr, va, wv, cv))


def _default_model():
    try:
        from .model import DEFAULT_MODEL
    except ImportError:
        from model import DEFAULT_MODEL
    return dict(DEFAULT_MODEL)


def load_checkpoint(path, device=None):
    device = device or get_device()
    ck = torch.load(path, map_location="cpu", weights_only=False)
    wv, cv = D.Vocab({}, 1), D.Vocab({}, 1)
    wv.itos, cv.itos = ck["vocab"], ck["chars"]
    wv.stoi = {w: i for i, w in enumerate(wv.itos)}
    cv.stoi = {c: i for i, c in enumerate(cv.itos)}
    model = build_model(ck["cfg"], wv, cv)
    model.load_state_dict(ck["state_dict"])
    return model.to(device).eval(), ck["cfg"], wv, cv


def _parse_set(items):
    """Parse key=json_value overrides. Unknown keys and values whose type doesn't match the default
    (e.g. a bool flag given a string) are errors, so a typo can never silently change an experiment."""
    defaults = {**DEFAULT_TRAIN, **_default_model()}
    out = {}
    for it in items or []:
        k, v = it.split("=", 1)
        if k not in defaults:
            raise SystemExit(f"--set: unknown config key {k!r}")
        try:
            val = json.loads(v)
        except json.JSONDecodeError:
            val = v
        d = defaults[k]
        ok = (d is None or val is None or isinstance(val, type(d))
              or (isinstance(d, float) and isinstance(val, int) and not isinstance(val, bool))
              or (isinstance(d, tuple) and isinstance(val, list)))
        if isinstance(d, bool) and not isinstance(val, bool):
            ok = False
        if not ok:
            raise SystemExit(f"--set: {k}={v!r} has type {type(val).__name__}, expected {type(d).__name__}")
        out[k] = val
    return out


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--name", default="main")
    ap.add_argument("--seed", type=int, default=1)
    ap.add_argument("--smoke", action="store_true", help="200 examples, 2 epochs, no runs.csv entry")
    ap.add_argument("--set", nargs="*", help="config overrides key=json_value")
    args = ap.parse_args()
    cfg = _parse_set(args.set)
    if args.smoke:
        cfg.update(epochs=2, patience=99)
    res = train(cfg, seed=args.seed, name=args.name, limit=200 if args.smoke else None,
                log=lambda s: print(s, flush=True), save=not args.smoke)
    if args.smoke:
        h = res["history"]
        assert h[-1]["train_loss"] < h[0]["train_loss"], "smoke test: loss did not decrease"
        tr, va, wv, cv = res["data"]
        model = res["model"]
        batch = to_device(D.collate(va[:4]), get_device())
        with torch.no_grad():
            model.eval()
            o = model(batch, return_attn=True)
        B, T = batch["t_words"].shape
        J = batch["q_words"].shape[1]
        assert o["S"].shape == (B, T, J) and o["c2q"].shape == (B, T, J)
        if o["q2c"] is not None:
            assert o["q2c"].shape == (B, T)
        if o["self_attn"] is not None:
            assert o["self_attn"].shape == (B, T, T)
        assert torch.allclose(o["c2q"].sum(-1), torch.ones(B, T, device=o["c2q"].device), atol=1e-4)
        print(f"SMOKE OK: loss {h[0]['train_loss']:.3f} -> {h[-1]['train_loss']:.3f}; "
              f"attn shapes S{tuple(o['S'].shape)} self{tuple(o['self_attn'].shape) if o['self_attn'] is not None else None}")
    sys.exit(0)


Writing src/train.py


In [10]:
%%writefile src/rerank.py
"""Deterministic lightweight reranking for TweetBiDAF Top-K spans.

The base QA model proposes spans.  A linear pairwise logistic model then
reorders them using fixed, interpretable features.  Validation gold answers
are never used to fit the reranker.
"""
import math
import re

import numpy as np
import torch

try:
    from . import data as D
    from .evaluate import normalize_answer, score_example
    from .train import to_device
except ImportError:
    import data as D
    from evaluate import normalize_answer, score_example
    from train import to_device


QUESTION_TYPES = ("what", "who", "how", "where", "why", "which", "when", "other")
REGIONS = ("body", "byline", "cross")
BASE_FEATURES = (
    "base_score", "score_margin", "reciprocal_rank", "start_logp", "end_logp",
    "length", "length_fraction", "start_fraction", "end_fraction", "distance_from_end",
    "question_overlap", "question_jaccard", "context_overlap", "context_jaccard",
    "starts_word", "ends_word", "contains_mention", "contains_url", "contains_number",
)
FEATURE_NAMES = (BASE_FEATURES
                 + tuple(f"qtype_{x}" for x in QUESTION_TYPES)
                 + tuple(f"region_{x}" for x in REGIONS))


def _tokens(text):
    return normalize_answer(text).split()


def _overlap(a, b):
    a, b = set(_tokens(a)), set(_tokens(b))
    if not a or not b:
        return 0.0, 0.0
    inter = len(a & b)
    return inter / len(a), inter / len(a | b)


def span_region(ex, start, end):
    boundary = ex["byline_from"]
    if boundary is None or end < boundary:
        return "body"
    if start >= boundary:
        return "byline"
    return "cross"


@torch.no_grad()
def topk_candidates(model, examples, device, top_k=10, batch_size=64, max_span_len=15):
    """Return Top-K legal spans and decomposed base scores for each example."""
    model.eval()
    records = []
    for batch in D.make_loader(examples, batch_size=batch_size):
        meta = batch["meta"]
        out = model(to_device(batch, device))
        start = torch.log_softmax(out["start_logits"], -1)
        end = torch.log_softmax(out["end_logits"], -1)
        padded = start.size(1)
        positions = torch.arange(padded, device=start.device)
        span_width = positions.unsqueeze(0) - positions.unsqueeze(1)
        legal_band = (span_width >= 0) & (span_width < max_span_len)
        combined = start.unsqueeze(2) + end.unsqueeze(1)
        combined = combined.masked_fill(~legal_band.unsqueeze(0), -1e30)
        for b, ex in enumerate(meta):
            length = len(ex["t_tokens"])
            legal_count = sum(min(max_span_len, length - i) for i in range(length))
            values, flat_indices = combined[b].reshape(-1).topk(min(top_k, legal_count))
            spans = []
            for value, flat in zip(values.tolist(), flat_indices.tolist()):
                i, j = flat // padded, flat % padded
                spans.append(dict(
                    start=i, end=j, text=D.span_text(ex, i, j),
                    start_logp=float(start[b, i]), end_logp=float(end[b, j]),
                    base_score=value, region=span_region(ex, i, j),
                ))
            records.append(dict(row_id=ex["row_id"], qid=ex["qid"], candidates=spans))
    return records


def candidate_features(ex, candidate, rank, top_score):
    """Fixed numeric feature vector; no corpus statistics or fitted parameters."""
    i, j = candidate["start"], candidate["end"]
    n = max(1, len(ex["t_tokens"])); width = j - i + 1
    question = ex["question"]; text = candidate["text"]
    left, right = max(0, i - 3), min(n - 1, j + 3)
    context = D.span_text(ex, left, right)
    q_overlap, q_jaccard = _overlap(question, text)
    c_overlap, c_jaccard = _overlap(question, context)
    raw_start = ex["t_tokens"][i]["raw"] if ex["t_tokens"] else ""
    raw_end = ex["t_tokens"][j]["raw"] if ex["t_tokens"] else ""
    values = [
        candidate["base_score"], candidate["base_score"] - top_score, 1.0 / rank,
        candidate["start_logp"], candidate["end_logp"], width, width / n,
        i / n, j / n, (n - 1 - j) / n,
        q_overlap, q_jaccard, c_overlap, c_jaccard,
        float(bool(re.search(r"\w", raw_start))), float(bool(re.search(r"\w", raw_end))),
        float("@" in text), float(bool(re.search(r"https?://|www\.", text, re.I))),
        float(bool(re.search(r"\d", text))),
    ]
    qtype = D._question_word(question)
    values.extend(float(qtype == x) for x in QUESTION_TYPES)
    values.extend(float(candidate["region"] == x) for x in REGIONS)
    return np.asarray(values, dtype=np.float64)


def feature_matrix(examples, records):
    by_id = {ex["row_id"]: ex for ex in examples}
    matrices = []
    for record in records:
        ex = by_id[record["row_id"]]
        top_score = record["candidates"][0]["base_score"]
        matrices.append(np.stack([candidate_features(ex, c, rank, top_score)
                                  for rank, c in enumerate(record["candidates"], 1)]))
    return matrices


def fit_pairwise_reranker(examples, records, regularization=1.0):
    """Fit one standardized linear ranking model from pairwise preferences.

    For each question, the candidate with maximum gold ROUGE-L is preferred to
    candidates with strictly lower ROUGE-L.  Reversed pairs balance the binary
    labels exactly.  `regularization` is fixed to 1.0 in the experiment.
    """
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler

    by_id = {ex["row_id"]: ex for ex in examples}
    matrices = feature_matrix(examples, records)
    pairs, labels = [], []
    useful = 0
    for record, matrix in zip(records, matrices):
        ex = by_id[record["row_id"]]
        quality = np.asarray([score_example(c["text"], ex["answers"], ("rougeL",))["rougeL"]
                              for c in record["candidates"]])
        best = int(quality.argmax())
        made_pair = False
        for other in range(len(quality)):
            if quality[best] <= quality[other] + 1e-12:
                continue
            diff = matrix[best] - matrix[other]
            pairs.extend((diff, -diff)); labels.extend((1, 0)); made_pair = True
        useful += int(made_pair)
    if not pairs:
        raise ValueError("no strict candidate preferences available for reranker training")
    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(C=1.0 / regularization, solver="lbfgs", max_iter=1000,
                           random_state=4012),
    )
    model.fit(np.stack(pairs), np.asarray(labels))
    return model, dict(pair_rows=len(pairs), useful_questions=useful,
                       total_questions=len(records), regularization=regularization)


def fit_lambdamart_reranker(examples, records):
    """Fit one conservative LambdaMART model with a fixed shallow tree setup.

    Relevance is gold ROUGE-L quantised to five grades (0..4).  Query groups
    are passed explicitly, so the objective learns ordering within each
    question rather than a global answer-quality scale.
    """
    from lightgbm import LGBMRanker

    by_id = {ex["row_id"]: ex for ex in examples}
    matrices = feature_matrix(examples, records)
    labels, groups = [], []
    for record in records:
        ex = by_id[record["row_id"]]
        quality = [score_example(c["text"], ex["answers"], ("rougeL",))["rougeL"]
                   for c in record["candidates"]]
        labels.extend(min(4, int(round(4 * value))) for value in quality)
        groups.append(len(record["candidates"]))
    model = LGBMRanker(
        objective="lambdarank", metric="ndcg", n_estimators=150,
        learning_rate=0.05, num_leaves=15, max_depth=4,
        min_child_samples=50, reg_lambda=1.0,
        random_state=4012, deterministic=True, force_col_wise=True,
        verbosity=-1, n_jobs=-1,
    )
    model.fit(np.concatenate(matrices), np.asarray(labels, dtype=np.int32), group=groups)
    return model, dict(candidate_rows=len(labels), total_questions=len(records),
                       relevance_levels=5, fixed_trees=150, fixed_max_depth=4)


def rerank(model, examples, records):
    matrices = feature_matrix(examples, records)
    predictions, selections = [], []
    for record, matrix in zip(records, matrices):
        scores = (model.decision_function(matrix) if hasattr(model, "decision_function")
                  else model.predict(matrix))
        index = int(np.argmax(scores))
        predictions.append(record["candidates"][index]["text"])
        selections.append(dict(row_id=record["row_id"], qid=record["qid"],
                               selected_rank=index + 1, rerank_score=float(scores[index]),
                               **record["candidates"][index]))
    return predictions, selections


def linear_weights(model):
    """Return coefficients in standardized feature space for interpretation."""
    classifier = model.named_steps["logisticregression"]
    return sorted(zip(FEATURE_NAMES, classifier.coef_[0].tolist()),
                  key=lambda item: abs(item[1]), reverse=True)


def tree_importances(model):
    """Return fixed LambdaMART split importances for descriptive reporting."""
    return sorted(zip(FEATURE_NAMES, model.feature_importances_.tolist()),
                  key=lambda item: item[1], reverse=True)


Writing src/rerank.py


## Load and verify required fold artifacts

In [11]:
from src import data as D,train as T,rerank as R
from src.evaluate import evaluate,score_example
from lightgbm import LGBMRanker
from sklearn.model_selection import StratifiedGroupKFold
train_rows,val_rows=D.get_splits()
assert D.TEST_LOCKED
with zipfile.ZipFile(OOFZIP) as z:
    prior=json.loads(z.read('oof_reranker_results/oof_reranker_summary.json'))
with zipfile.ZipFile(SOURCE) as z:
    rid=prior['source_run']; full_path=WORK/(rid+'.pt')
    full_path.write_bytes(z.read('attention_variant_checkpoints/'+rid+'.pt'))
fold_runs=list(csv.DictReader((FOLD_ROOT/'oof_base_results/runs.csv').open(encoding='utf-8-sig')))
fold_paths={}
for fold in (1,2,3):
    matches=[r for r in fold_runs if r['model']==f'OOFBaseFold{fold}']
    assert matches, f'Missing fold {fold} run'
    p=FOLD_ROOT/'oof_base_checkpoints'/(matches[-1]['run_id']+'.pt')
    assert p.exists(),f'Missing {p}; restore Step 10 checkpoint. No automatic retraining.'
    assert (FOLD_ROOT/f'fold{fold}_oof_top10.jsonl').exists()
    fold_paths[fold]=p
labels=np.array([D._question_word(r['question']) for r in train_rows])
groups=np.array([r['tweet'] for r in train_rows],dtype=object)
folds=list(StratifiedGroupKFold(n_splits=3,shuffle=True,random_state=4012).split(np.zeros(len(train_rows)),labels,groups))
def prepare(rows,cfg,wv,cv):
    ex=D.preprocess(rows,dict(use_normaliser=cfg['use_normaliser'],max_span_len=cfg['max_span_len'],tau=cfg['tau'],soft_tau=cfg.get('soft_tau')))
    D.encode(ex,wv,cv); return ex
def read_jsonl(path): return [json.loads(x) for x in path.read_text(encoding='utf-8').splitlines()]
REP_NAMES=['question_span_mean_cos','question_context_mean_cos','question_left_mean_cos','question_right_mean_cos','question_to_span_maxcos_mean','span_to_question_maxcos_mean']


## Representation extraction

In [12]:
import torch.nn.functional as F
def representation_features(h,u,start,end,window=3):
    # h/u are trimmed to real tokens: no padded vectors in pooling.
    assert 0<=start<=end<len(h) and len(u)>0
    q=u.mean(0); span=h[start:end+1]
    def cosine(v):
        return float(F.cosine_similarity(q[None],v.mean(0)[None],dim=-1,eps=1e-8)) if len(v) else 0.
    pair=F.normalize(u,dim=-1,eps=1e-8)@F.normalize(span,dim=-1,eps=1e-8).T
    values=[cosine(span),cosine(h[max(0,start-window):min(len(h),end+window+1)]),
            cosine(h[max(0,start-window):start]),cosine(h[end+1:min(len(h),end+window+1)]),
            float(pair.max(1).values.mean()),float(pair.max(0).values.mean())]
    assert len(values)==len(REP_NAMES) and np.isfinite(values).all()
    return values

# Small deterministic checks run before expensive fold inference.
_h=torch.tensor([[1.,0.],[0.,1.],[1.,0.]])
_u=torch.tensor([[1.,0.]])
_v=representation_features(_h,_u,0,0)
assert abs(_v[0]-1.)<1e-6 and _v[2]==0. and _v[4]==1. and _v[5]==1.
assert representation_features(_h,_u,2,2)[3]==0.
assert np.isfinite(representation_features(torch.zeros(2,2),torch.zeros(1,2),0,1)).all()
print('Representation feature sanity checks passed')

@torch.no_grad()
def enrich(model,examples,records,cfg):
    model.eval(); lookup={r['row_id']:r for r in records}
    assert len(lookup)==len(examples)==len(records)
    assert set(lookup)=={e['row_id'] for e in examples}
    # Verify checkpoint/candidate provenance before attaching representations.
    generated=R.topk_candidates(model,examples,DEVICE,top_k=10,batch_size=64,max_span_len=cfg['max_span_len'])
    for g in generated:
        old=lookup[g['row_id']]
        assert [(c['start'],c['end']) for c in g['candidates']]==[(c['start'],c['end']) for c in old['candidates']], 'Candidate/checkpoint mismatch'
        assert np.allclose([c['base_score'] for c in g['candidates']],[c['base_score'] for c in old['candidates']],atol=2e-4,rtol=1e-4), 'Candidate score mismatch'
    for batch in D.make_loader(examples,batch_size=64):
        b=T.to_device(batch,DEVICE)
        # Same shared encoder and embedding as model.forward, frozen in eval mode.
        h=model.drop(model.encoder(model.embed(b['t_words'],b['t_chars'],b['t_flags']),b['t_mask']))
        u=model.drop(model.encoder(model.embed(b['q_words'],b['q_chars'],b['q_flags']),b['q_mask']))
        for i,e in enumerate(batch['meta']):
            ht=h[i,:int(b['t_mask'][i].sum())]; ut=u[i,:int(b['q_mask'][i].sum())]
            for c in lookup[e['row_id']]['candidates']:
                c['representation_features']=representation_features(ht,ut,c['start'],c['end'])
    return [lookup[e['row_id']] for e in examples]

def checkpoint_hash(path):
    digest=hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda:stream.read(1024*1024),b''): digest.update(block)
    return digest.hexdigest()
def enriched_cache(tag,path,rows,records):
    # Cache includes model and candidate fingerprints; stale values are rejected.
    key=hashlib.sha256((checkpoint_hash(path)+json.dumps(records,sort_keys=True)+json.dumps(REP_NAMES)+str(3)).encode()).hexdigest()
    cache=BACKUP/(tag+'_representations.json')
    model,cfg,wv,cv=T.load_checkpoint(path,DEVICE)
    examples=prepare(rows,cfg,wv,cv)
    payload=json.loads(cache.read_text(encoding='utf-8')) if cache.exists() else None
    if payload and payload.get('fingerprint')==key:
        enriched=payload['records']; print('CACHE',tag)
        assert [r['row_id'] for r in enriched]==[e['row_id'] for e in examples]
        assert all(len(c['representation_features'])==len(REP_NAMES) and np.isfinite(c['representation_features']).all() for r in enriched for c in r['candidates'])
    else:
        enriched=enrich(model,examples,records,cfg)
        cache.write_text(json.dumps(dict(fingerprint=key,records=enriched)),encoding='utf-8')
        print('SAVED representations',tag,len(enriched))
    del model; torch.cuda.empty_cache()
    return examples,enriched


Representation feature sanity checks passed


## Extract fold-safe training and validation features

In [13]:
oof=[]; train=[]
for fold,(fit_idx,hold_idx) in enumerate(folds,1):
    assert not (set(groups[fit_idx])&set(groups[hold_idx]))
    rows=[train_rows[i] for i in hold_idx]
    records=read_jsonl(FOLD_ROOT/f'fold{fold}_oof_top10.jsonl')
    examples,enriched=enriched_cache(f'fold{fold}',fold_paths[fold],rows,records)
    train.extend(examples); oof.extend(enriched)
assert len(train)==len(oof)==len(train_rows)
assert len({e['row_id'] for e in train})==len(train_rows)
order=np.argsort([e['row_id'] for e in train]); train=[train[i] for i in order]; oof=[oof[i] for i in order]
model,cfg,wv,cv=T.load_checkpoint(full_path,DEVICE)
val=prepare(val_rows,cfg,wv,cv)
records=R.topk_candidates(model,val,DEVICE,top_k=10,batch_size=64,max_span_len=cfg['max_span_len'])
del model; torch.cuda.empty_cache()
val,val_records=enriched_cache('validation',full_path,val_rows,records)
print('OOF features ready',len(train),'validation',len(val))


SAVED representations fold1 3229
SAVED representations fold2 3158
SAVED representations fold3 3183
SAVED representations validation 1076
OOF features ready 9570 validation 1076


## Fixed reranker definitions

In [14]:
def matrices(examples,records,enhanced):
    base=R.feature_matrix(examples,records)
    if not enhanced: return base
    return [np.column_stack((x,np.asarray([c['representation_features'] for c in r['candidates']]))) for x,r in zip(base,records)]


def fit_tree(examples,records,enhanced):
    xs=matrices(examples,records,enhanced); labels=[]
    lookup={e['row_id']:e for e in examples}
    for r in records:
        labels.extend(min(4,int(round(4*score_example(c['text'],lookup[r['row_id']]['answers'],('rougeL',))['rougeL']))) for c in r['candidates'])
    m=LGBMRanker(objective='lambdarank',n_estimators=150,learning_rate=.05,num_leaves=15,max_depth=4,min_child_samples=50,reg_lambda=1.,random_state=4012,deterministic=True,force_col_wise=True,verbosity=-1,n_jobs=-1)
    m.fit(np.concatenate(xs),np.array(labels),group=[len(x) for x in xs]); return m

def predict_tree(m,examples,records,enhanced):
    return [r['candidates'][int(np.argmax(m.predict(x)))]['text'] for r,x in zip(records,matrices(examples,records,enhanced))]

## Fit and evaluate on unchanged candidates

In [15]:
# Two predefined configurations, unchanged tree hyperparameters; no tuning.
predictions={'BaseTop1':[r['candidates'][0]['text'] for r in val_records]}
importances={}
for enhanced in (False,True):
    name='RepresentationOOF' if enhanced else 'OriginalOOF'
    model=fit_tree(train,oof,enhanced)
    predictions[name]=predict_tree(model,val,val_records,enhanced)
    joblib.dump(model,OUT/(name+'.joblib'))
    names=list(R.FEATURE_NAMES)+(REP_NAMES if enhanced else [])
    importances[name]=sorted(zip(names,model.feature_importances_.tolist()),key=lambda x:x[1],reverse=True)
golds=[e['answers'] for e in val]
scores={name:evaluate(p,golds) for name,p in predictions.items()}
for metric in ('bleu1','meteor','rougeL','em','f1'):
    assert abs(scores['OriginalOOF'][metric]-prior['scores']['OOFLambdaMART'][metric])<1e-4, 'Original OOF failed reproduction'
lines=['| System | BLEU-1 | METEOR | ROUGE-L | EM | F1 |','|---|---:|---:|---:|---:|---:|']
for name,s in scores.items(): lines.append('| '+name+' | '+' | '.join(f'{s[m]:.2f}' for m in ('bleu1','meteor','rougeL','em','f1'))+' |')
display(Markdown('\n'.join(lines)))
for name,p in predictions.items():
    (OUT/(name+'_predictions.json')).write_text(json.dumps([dict(row_id=e['row_id'],pred=a) for e,a in zip(val,p)]),encoding='utf-8')
(OUT/'feature_schema.json').write_text(json.dumps(dict(base=list(R.FEATURE_NAMES),representation=REP_NAMES,encoder='shared encoder before BiDAF attention',window=3),indent=2),encoding='utf-8')
ex=val


| System | BLEU-1 | METEOR | ROUGE-L | EM | F1 |
|---|---:|---:|---:|---:|---:|
| BaseTop1 | 39.06 | 33.69 | 44.14 | 27.32 | 44.22 |
| OriginalOOF | 40.60 | 34.26 | 45.56 | 28.90 | 45.73 |
| RepresentationOOF | 40.91 | 34.48 | 45.87 | 29.09 | 45.99 |

## Tweet bootstrap and save

In [16]:
# Cluster bootstrap: all questions from a tweet are resampled together.
from collections import defaultdict
groups=defaultdict(list)
for i,e in enumerate(ex): groups[e['tweet']].append(i)
indices=list(groups.values()); rng=np.random.default_rng(4012); intervals={}
lines=['| Comparison | Metric | Delta | Tweet bootstrap 95% CI |','|---|---|---:|---:|']
for a,b in [('OriginalOOF','BaseTop1'),('RepresentationOOF','OriginalOOF')]:
    intervals[a+'_vs_'+b]={}
    for metric in ('rougeL','f1','em'):
        d=np.array([score_example(p,g,(metric,))[metric]-score_example(q,g,(metric,))[metric] for p,q,g in zip(predictions[a],predictions[b],golds)])
        sums=np.array([d[ix].sum() for ix in indices]); sizes=np.array([len(ix) for ix in indices]); means=[]
        for _ in range(5000):
            chosen=rng.integers(0,len(indices),len(indices)); means.append(100*sums[chosen].sum()/sizes[chosen].sum())
        lo,hi=np.quantile(means,[.025,.975]); delta=float(d.mean()*100)
        intervals[a+'_vs_'+b][metric]=dict(delta=delta,low=float(lo),high=float(hi))
        lines.append(f'| {a} vs {b} | {metric} | {delta:+.2f} | [{lo:+.2f}, {hi:+.2f}] |')
display(Markdown('\n'.join(lines)))
summary=dict(scores=scores,paired_tweet_bootstrap=intervals,n_questions=len(ex),n_tweets=len(indices),test_accessed=False,note='Validation-only fixed-candidate OOF representation study. Intervals do not correct for previous validation-based model selection.')
summary['feature_importances']=importances
summary['source_run']=prior['source_run']
(OUT/'summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
(OUT/'report.md').write_text('\n'.join(lines)+'\n\n'+summary['note'],encoding='utf-8')
archive=Path(shutil.make_archive(str(WORK/'representation_reranker_artifacts'),'zip',OUT))
shutil.copy2(archive,BACKUP/archive.name)
print('SAVED',BACKUP/archive.name)
print('Completed: validation only, no base-model training and no test evaluation.')


| Comparison | Metric | Delta | Tweet bootstrap 95% CI |
|---|---|---:|---:|
| OriginalOOF vs BaseTop1 | rougeL | +1.42 | [+0.29, +2.57] |
| OriginalOOF vs BaseTop1 | f1 | +1.50 | [+0.40, +2.68] |
| OriginalOOF vs BaseTop1 | em | +1.58 | [+0.38, +2.83] |
| RepresentationOOF vs OriginalOOF | rougeL | +0.30 | [-0.35, +0.96] |
| RepresentationOOF vs OriginalOOF | f1 | +0.27 | [-0.39, +0.94] |
| RepresentationOOF vs OriginalOOF | em | +0.19 | [-0.65, +1.00] |

SAVED /content/drive/MyDrive/CITS4012_A2_representation_reranker/representation_reranker_artifacts.zip
Completed: validation only, no base-model training and no test evaluation.
